<a href="https://colab.research.google.com/github/Perine23Dev/analyse-entreprises-emploi-republique-du-congo/blob/main/projet_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importation

In [14]:
import io
import pandas as pd
import numpy as np
import matplotlib as plt
from google.colab import auth
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

auth.authenticate_user()
creds, _ = default()
service = build("drive", "v3", credentials=creds)

file_id = "18zTTJ434DgPIt9JCoXujQ6HgvXpY0zKg"

requete = service.files().get_media(fileId=file_id)
tampon = io.BytesIO()
telechargement = MediaIoBaseDownload(tampon, requete)

fini = False
while not fini:
    _, fini = telechargement.next_chunk()

tampon.seek(0)
df = pd.read_excel(tampon)


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


**Exploration de base**

In [25]:
dimension=df.shape
print("la dimension du data frame est : ", dimension)

la dimension du data frame est :  (18787, 20)


Conclusion : ce jeu de données contient 18787 lignes et 20 colonnes

In [19]:
#affichage des 5 premières lignes du dataframe
df.head()

Affichage des 5 premieres lignes du data frame


,Année création,Département,Raison sociale établissement principal,Activité déclarée,Activité au sens de la nomenclature OHADA,Forme juridique,Statut juridique,Catégorie,Genre,Age,Nationalité,Branche,Secteur,capital,part_capital_etranger,salarie,nbre_empl_congolais,chiffre_aff_mo_annuel,chiffre_aff_mo_annuel_1,chiffre_aff_mo_annuel_2
0,2023,Pointe-Noire,EL SHADDAI,MERCERIE,Commerce de détail en magasin spécialisé de te...,EI,Entreprise individuelle,TPE-PE,Femme,45.0,Congolaise,Commerce,Tertiaire,500000,0.0,1.0,1.0,1000000,1500000,2000000.0
1,2023,Niari,BADJEL,Commerce de détail en magasin spécialisé des p...,Commerce de détail en magasin spécialisé des p...,EI,Entreprise individuelle,TPE,Homme,49.0,Mauritanienne,Commerce,Tertiaire,NaN,NaN,1.0,0.0,1500000,2000000,2500000.0
2,2015,Pointe-Noire,HIRAM SERVICES,MISE A DISPOSITION DE PERSONNEL,activités liées aux ressources humaines,SARLU,Société de personnes,TPE-PE,Homme,43.0,Congolaise,Services aux entreprises,Tertiaire,0,0.0,1.0,1.0,0,0,0.0
3,2015,Pointe-Noire,HB,VENTE D'ELECTROMENAGER,commerce de détail en magasin spécialisé d’art...,EI,Entreprise individuelle,TPE-PE,Homme,40.0,Malienne,Commerce,Tertiaire,0,0.0,4.0,0.0,0,0,0.0
4,2026,Niari,BADJEL,Commerce de détail en magasin spécialisé de pr...,Commerce de détail en magasin spécialisé de pr...,EI,Entreprise individuelle,TPE,Homme,49.0,Mauritanienne,Commerce,Tertiaire,NaN,NaN,1.0,0.0,1500000,2000000,2500000.0


Les cinq premières lignes montrent une diversité d’entreprises en termes d’année de création, d’activité, de forme juridique, de nationalité et de taille. Les entreprises observées appartiennent principalement au secteur tertiaire, notamment au commerce et aux services. On remarque également la présence de valeurs manquantes dans certaines variables, notamment le capital et la part de capital étranger. La raison sociale BADJEL apparaît deux fois, ce qui devra être vérifié lors de l’analyse des doublons. Enfin, les données contiennent des informations sur les salariés et le chiffre d’affaires permettant d’étudier ultérieurement la situation et l’évolution des entreprises.

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18787 entries, 0 to 18786
Data columns (total 20 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Année création                             18787 non-null  int64  
 1   Département                                18787 non-null  object 
 2   Raison sociale établissement principal     18787 non-null  object 
 3   Activité déclarée                          18777 non-null  object 
 4   Activité au sens de la nomenclature OHADA  18683 non-null  object 
 5   Forme juridique                            18787 non-null  object 
 6   Statut juridique                           18787 non-null  object 
 7   Catégorie                                  5545 non-null   object 
 8   Genre                                      18675 non-null  object 
 9   Age                                        18455 non-null  float64
 10  Nationalité           

La base de données comprend 18 787 observations et 20 variables, couvrant les caractéristiques démographiques, juridiques et économiques des entreprises. Elle contient 13 variables qualitatives et 7 variables quantitatives. La plupart des variables sont correctement renseignées, mais certaines présentent des données manquantes, notamment la variable Catégorie, dont environ 70,5 % des valeurs sont absentes. Les variables capital et chiffre d'affaires nécessitent également une vérification de leur format, car elles sont enregistrées comme variables textuelles alors qu'elles représentent a priori des grandeurs quantitatives. Ces éléments devront être traités avant l'analyse statistique.

In [21]:
#affichage des premières statistiques du dataframes
df.describe()

,Année création,Age,part_capital_etranger,salarie,nbre_empl_congolais,chiffre_aff_mo_annuel_1,chiffre_aff_mo_annuel_2
count,18787.000000,18455.000000,1.713600e+04,18784.000000,18724.000000,1.878700e+04,1.878700e+04
mean,2023.647948,40.550257,2.230160e+06,21.979291,2.457862,8.991517e+07,9.798952e+07
std,3.398257,21.559952,1.480841e+08,2614.726019,13.735897,5.934576e+09,6.022792e+09
min,1949.000000,-174.000000,0.000000e+00,0.000000,-2.000000,0.000000e+00,0.000000e+00
25%,2023.000000,32.000000,0.000000e+00,1.000000,1.000000,1.000000e+06,1.000000e+06
50%,2025.000000,38.000000,0.000000e+00,1.000000,1.000000,2.500000e+06,3.000000e+06
75%,2025.000000,47.000000,0.000000e+00,2.000000,2.000000,1.000000e+07,1.000000e+07
max,2028.000000,1043.000000,1.269362e+10,358358.000000,536.000000,8.000000e+11,8.100000e+11


Le tableau des statistiques descriptives montre que les variables quantitatives présentent des niveaux et des dispersions différents. L'âge moyen est de 40,55 ans, avec une médiane de 38 ans et un intervalle interquartile compris entre 32 et 47 ans. Le nombre de salariés présente une moyenne de 21,98 contre une médiane de 1, tandis que 75 % des observations comptent au maximum 2 salariés. Cette différence entre la moyenne et la médiane traduit une importante dispersion des effectifs. Concernant le chiffre d'affaires, la moyenne est de 89,92 millions pour la première période et de 97,99 millions pour la seconde, alors que les médianes sont respectivement de 2,5 millions et 3 millions. L'écart entre ces valeurs indique également une forte dispersion du chiffre d'affaires. Enfin, certaines valeurs minimales et maximales, notamment pour l'âge, le nombre d'employés et le chiffre d'affaires, apparaissent particulièrement élevées ou incohérentes et devront être vérifiées.

In [23]:
#verifications des valeurs manquantes sur l'ensembler du dataframes
df.isnull().sum()

,0
Année création,0
Département,0
Raison sociale établissement principal,0
Activité déclarée,10
Activité au sens de la nomenclature OHADA,104
Forme juridique,0
Statut juridique,0
Catégorie,13242
Genre,112
Age,332


In [29]:
#calcul du pourcentage des valeurs manquantes
(df.isnull().sum() / len(df) * 100).round(2)


,0
Année création,0.00
Département,0.00
Raison sociale établissement principal,0.00
Activité déclarée,0.05
Activité au sens de la nomenclature OHADA,0.55
Forme juridique,0.00
Statut juridique,0.00
Catégorie,70.48
Genre,0.60
Age,1.77


L’analyse des valeurs manquantes montre que la majorité des variables sont bien renseignées. La variable « Catégorie » présente toutefois un taux élevé de valeurs manquantes (70,48 %), suivie du capital (10,82 %) et de la part du capital étranger (8,79 %). Les autres variables présentent des taux de valeurs manquantes faibles, généralement inférieurs à 3 %. Ces variables devront donc faire l’objet d’une attention particulière lors du nettoyage des données.




In [24]:
df.duplicated().sum()

np.int64(0)

Aucune ligne n'est répétée, donc les données ne contiennent pas de doublons.